In [9]:
import os
import requests
from pinecone import Pinecone

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
INDEX_NAME = "sailok-laws"

EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2:free"


def get_embeddings(texts, api_key, model=EMBEDDING_MODEL):
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "input": texts,
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        json=payload,
        headers=headers,
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()
    return [item["embedding"] for item in data["data"]]


pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)


def search_and_show_full_metadata(query, top_k=3):
    """Search Pinecone and print full metadata for each match."""
    query_embedding = get_embeddings([query], OPENROUTER_API_KEY)[0]

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
    )

    print(f"\nQuery: {query}")
    print("=" * 80)

    for i, match in enumerate(results["matches"], 1):
        print(f"\nResult {i}")
        print(f"Score: {match['score']:.4f}")
        print(f"ID: {match['id']}")
        print("-" * 40)

        metadata = match.get("metadata", {})
        if not metadata:
            print("No metadata returned")
            continue

        for key, value in metadata.items():
            print(f"{key}: {value}")

    return results


# Test it
search_and_show_full_metadata("Can a 17 year old get a loan?", top_k=2)


Query: Can a 17 year old get a loan?

Result 1
Score: 0.1594
ID: Code of Conduct under Law 4224/2013_Sections A-K, Annexes 1-3_2
----------------------------------------
article: Sections A-K, Annexes 1-3
confidence: 0.98
keywords: cooperating borrower, loan arrears, forbearance, Code of Conduct, non-performing loans, Arrears Resolution Procedure, Bank of Greece, Standardised Statement of Financial Information
law_name: Code of Conduct under Law 4224/2013
law_number: Decision 116/25.08.2014
paragraph: Multiple
priority: HIGH
rules_count: 9.0
summary: This decision establishes the Greek Code of Conduct for managing loan arrears, setting out strict timelines and procedures for banks to offer restructuring options and defining the obligations of a 'cooperating borrower.'[cite: 2]
text_preview: Bank of Greece Credit and Insurance Committee Decision No. 116/25.08.2014 introduces a Code of Conduct under Law 4224/2013 regarding the management of non-performing loans and loans in arrears[cite

{'matches': [{'id': 'Code of Conduct under Law 4224/2013_Sections A-K, Annexes '
                    '1-3_2',
              'metadata': {'article': 'Sections A-K, Annexes 1-3',
                           'confidence': 0.98,
                           'keywords': 'cooperating borrower, loan arrears, '
                                       'forbearance, Code of Conduct, '
                                       'non-performing loans, Arrears '
                                       'Resolution Procedure, Bank of Greece, '
                                       'Standardised Statement of Financial '
                                       'Information',
                           'law_name': 'Code of Conduct under Law 4224/2013',
                           'law_number': 'Decision 116/25.08.2014',
                           'paragraph': 'Multiple',
                           'priority': 'HIGH',
                           'rules_count': 9.0,
                           'summary': 'This decisi

In [10]:
def search_with_filter(query, top_k=3, filter=None):
    query_embedding = get_embeddings([query], OPENROUTER_API_KEY)[0]

    kwargs = {
        "vector": query_embedding,
        "top_k": top_k,
        "include_metadata": True,
    }

    if filter:
        kwargs["filter"] = filter

    results = index.query(**kwargs)

    for match in results["matches"]:
        print(match["id"], match["score"])
        print(match.get("metadata", {}))
        print()

    return results


search_with_filter(
    "loan eligibility for minors",
    top_k=2,
    filter={"priority": "HIGH"}
)

Directive on credit agreements for consumers relating to residential immovable property (Mortgage Credit Directive)_1-13_6 0.171651497
{'article': '1-13', 'confidence': 0.98, 'keywords': 'mortgage, credit agreements, consumer protection, creditworthiness assessment, ESIS, APRC, tying practices, bundling, foreign currency loans', 'law_name': 'Directive on credit agreements for consumers relating to residential immovable property (Mortgage Credit Directive)', 'law_number': '2014/17/EU', 'paragraph': 'Various', 'priority': 'HIGH', 'rules_count': 6.0, 'summary': 'This Directive sets EU-wide consumer protection standards for residential mortgage loans, requiring creditors to rigorously assess borrower creditworthiness, provide clear standardized pre-contractual information (ESIS), prohibit unfair tying practices, and enforce minimum staff competence.', 'text_preview': 'Directive 2014/17/EU of the European Parliament and of the Council of 4 February 2014 on credit agreements for consumers re

{'matches': [{'id': 'Directive on credit agreements for consumers relating to '
                    'residential immovable property (Mortgage Credit '
                    'Directive)_1-13_6',
              'metadata': {'article': '1-13',
                           'confidence': 0.98,
                           'keywords': 'mortgage, credit agreements, consumer '
                                       'protection, creditworthiness '
                                       'assessment, ESIS, APRC, tying '
                                       'practices, bundling, foreign currency '
                                       'loans',
                           'law_name': 'Directive on credit agreements for '
                                       'consumers relating to residential '
                                       'immovable property (Mortgage Credit '
                                       'Directive)',
                           'law_number': '2014/17/EU',
                         